# 🧠 LeetCode 739: Daily Temperatures

**Pattern:** Monotonic Stack

- **Created:** 2026-02-28
- **Focus:** Finding the Next Greater Element (NGE) in O(N)
- **Tags:** `array` | `stack` | `monotonic-stack` | `medium` | `citi-prep`

## 📖 Problem Statement

Given an array of integers `temperatures` represents the daily temperatures, return an array `answer` such that `answer[i]` is the number of days you have to wait after the $i^{th}$ day to get a warmer temperature.

If there is no future day for which this is possible, keep `answer[i] == 0` instead.

### Example 1:
- **Input:** `temperatures = [73,74,75,71,69,72,76,73]`
- **Output:** `[1,1,4,2,1,1,0,0]`

### Example 2:
- **Input:** `temperatures = [30,40,50,60]`
- **Output:** `[1,1,1,0]`

## 🧠 Mental Model & Intuition

This introduces the **Monotonic Stack** (a stack where elements are strictly increasing or decreasing).

**The "Waiting Room" Analogy:**
Imagine days are people standing in a waiting room holding a number (their temperature) and waiting for someone taller (warmer) to arrive.

- A `75` walks into the room. Nobody taller is there. He sits in the waiting room (Stack).
- A `71` walks in. He isn't taller than 75. He sits down next to him.
- A `69` walks in. Sitting.
- A `72` walks in! 

Suddenly, `72` looks at the waiting room from the entrance. He says, "Hey `69`, I'm taller than you. Get out!" Then he looks at `71`. "Hey `71`, I'm taller than you too. Get out!" 
He looks at `75`. "Oh, you're taller than me. I'll take a seat."

Every time someone was told to "Get out!" (popped from the stack), they were finally matched with their Next Greater Element. The difference in their arrival days (indexes) is the answer.

## 🐢 Brute Force Approach

For every single day, look at every single subsequent day to find the first one that is warmer.

```python
def dailyTemperaturesBrute(temperatures):
    res = [0] * len(temperatures)
    for i in range(len(temperatures)):
        for j in range(i + 1, len(temperatures)):
            if temperatures[j] > temperatures[i]:
                res[i] = j - i
                break
    return res
```
**Time:** $O(N^2)$ (Timeout on large datasets) | **Space:** $O(1)$ excluding output array.

In [ ]:
# Optimal Monotonic Stack Approach
def dailyTemperatures(temperatures: list[int]) -> list[int]:
    res = [0] * len(temperatures)
    stack = []  # Pair: [temp, index]
    
    for i, t in enumerate(temperatures):
        # While the stack has people sitting in the waiting room,
        # AND the person walking in the door (t) is warmer than the 
        # person closest to the door (stack[-1]), fire them from the waiting room.
        while stack and t > stack[-1][0]:
            stack_t, stack_i = stack.pop()
            # We finally found a warmer day for stack_i!
            # The wait time is the current day index minus their arrival day index.
            res[stack_i] = i - stack_i
            
        # After firing all colder people, the current person takes a seat.
        stack.append([t, i])
        
    return res

print("Wait times: ", dailyTemperatures([73,74,75,71,69,72,76,73]))

## ⏱️ Complexity Analysis

- **Time Complexity:** $O(N)$. Even with the `while` loop inside the `for` loop, you must realize that every single day is pushed onto the stack exactly *once*, and popped from the stack at most *once*. Thus, across all $N$ days, there are $2N$ stack operations. It averages to $O(1)$ amortized per day.
- **Space Complexity:** $O(N)$. In the worst-case scenario where temperatures are strictly decreasing `[100, 99, 98, 97 ...]`, nobody ever gets fired from the waiting room. The stack holds all $N$ elements.

## 🎤 Interview Q&A

### Q1: Is the stack monotonically increasing or decreasing?
**Answer:** In this specific implementation, because we only pop when the incoming element is *greater*, the elements left sitting in the stack MUST be strictly decreasing. A 75 is allowed to sit on top of an 80, but an 85 would kick both of them out before sitting down.

---

### Q2: Why don't we need a final pass to assign `0` to the elements left in the stack at the end?
**Answer:** Because we pre-filled the entire `res` array with `0`s at the very beginning (`res = [0] * len(temperatures)`). Any element remaining in the stack at the end of the `for` loop simply never found a warmer day, making `0` the precisely correct mathematical answer. This saves an unnecessary final loop cleanup.

## 📚 Key Terminology

| Term | Definition | Example |
|---|---|---|
| **Monotonic Stack** | A stack whose elements are ordered (strictly increasing or decreasing). | Keeping `[100, 90, 80]` |
| **NGE (Next Greater Element)** | The standard algorithmic classification for this exact pattern. | Finding the first warmer day. |
| **Amortized Analysis** | Averaging an occasional slow operation (the while loop) over all fast operations to prove linear scaling. | $O(N)$ vs $O(N^2)$ |

## 💼 The Citi Narrative

**Context:** Trade Rebound Timing (Market Microstructure).

**Scenario:** Quantitative analysts wanted to backtest a momentum strategy. When a bond's price cratered, exactly how many milliseconds did it take on average for the price to "rebound" back above the original crash threshold? Scanning billions of tick rows using cross-joins in Oracle (`price T2 > price T1`) timed out.

**Impact:** Rewriting the historical tick-analysis engine using a Monotonic Stack solved the Next Greater Element problem in a single linear $O(N)$ pass. Analyzing the rebound timing of 2 billion ticks dropped from a 4-hour batch job failure to a 45-second successful execution.

In [ ]:
# EXERCISE: Optimize space slightly
# Storing a pair [temp, index] works, but we can just store the index!
# You can look up the temperature anytime with `temperatures[index]`.
def dailyTemperaturesIdxOnly(temperatures):
    res = [0] * len(temperatures)
    stack = []  # Stores indices ONLY
    for i, t in enumerate(temperatures):
        while stack and t > temperatures[stack[-1]]:
            stack_i = stack.pop()
            res[stack_i] = i - stack_i
        stack.append(i)
    return res
    
print("Index-only variation result: ", dailyTemperaturesIdxOnly([30,40,50,60]))

## 🎯 Summary: Key Takeaways

### The Pattern
**Monotonic Stack** — Storing unresolved items (indices) in a stack until a resolving condition (larger value) arrives.

### When to Use It
- ✅ Next Greater Element (NGE) algorithmic matching.
- ✅ Identifying time-to-recovery / rebound latency in market feeds.
- ❌ **Don't use when:** Simply finding the global historical maximum.

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force | $O(N^2)$ | $O(1)$ |
| Optimal | $O(N)$ | $O(N)$ |

### Interview Confidence Checklist
- [ ] Can explain the brute force and why it fails
- [ ] Can state the pattern name and core insight in one sentence
- [ ] Can write the optimal solution from memory
- [ ] Can state time and space complexity with justification
- [ ] Can name at least one real-world / Citi application

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master **Monotonic Stack** and you've mastered one of the most common patterns in data engineering interviews. 🚀